In [88]:
import pandas as pd
import numpy as np
from common import *
from JDistance import CJDistance
from JSample import CJSample
import seaborn as sns
from tqdm.notebook import tqdm
from warnings import filterwarnings
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score, precision_score, recall_score, f1_score
filterwarnings("ignore") 
np.set_printoptions(suppress=True)
pd.set_option('display.float_format',lambda x : '%.8f' % x)
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
df = pd.read_csv("/data/paper4/pred/result.csv",index_col=0)
all_result = []
bar = tqdm(total=df.shape[0]-1)
for model,train_set,data_set,yt,yp in zip(df['model'],df['train'],df['test'],df['y_true'],df['y_pred']):
    y_true = json.loads(yt)
    y_pred = json.loads(yp)
    tmp = {}
    tmp['model'] = model
    if  not train_set in ['baseline','reserve']:
        tmp['train'] = "%02d"%(int(train_set))
    else:
        tmp['train'] = train_set
    if not data_set in ['baseline','reserve']:
        tmp['test'] = "%02d"%(int(data_set))
    else:
        tmp['test'] = data_set
    tmp['acccuracy'] = accuracy_score(y_true,y_pred)
    tmp['precision'] = precision_score(y_true,y_pred)
    tmp['recall'] = recall_score(y_true,y_pred)
    tmp['f1_score'] = f1_score(y_true,y_pred)
    all_result.append(tmp)
    bar.update(1)
df_result = pd.DataFrame(all_result)

  0%|          | 0/179 [00:00<?, ?it/s]

In [84]:
def GetHistogram(df1,df2,bin_count=100):
    df_tmp1 = df1.copy( deep = True )
    df_tmp2 = df2.copy( deep = True )
    df_tmp = pd.concat([df_tmp1,df_tmp2],ignore_index = False)
    df_tmp = CJSample.Preprocess(df_tmp).fillna(0)
    
    ret_df = pd.DataFrame()
    ret_df1 = pd.DataFrame()
    ret_df2 = pd.DataFrame()
    ret_s1 = pd.Series()
    ret_s2 = pd.Series()
    for key in df_tmp.keys():
        if key in ['label']:
            continue
        #使用平滑降噪
        if key in ['in_bytes','out_bytes','in_pkts','out_pkts','duration']:
            sli_window = df_tmp.shape[0]//bin_count
        else:
            sli_window = 1
        #sli_window = 9 #for demo
        df_tmp[key] = df_tmp[key].rolling(window=sli_window).mean()
        df_tmp[key] = df_tmp[key].fillna(0)
        #重新对其赋值
        tmp = df_tmp[key].head(df_tmp1.shape[0]+sli_window-1)        
        df_tmp1[key] = tmp.tail(df_tmp1.shape[0]).values
        tmp = df_tmp[key].tail(df_tmp2.shape[0])
        df_tmp2[key] = tmp
        #统一划分bin
        hist, bin_edges = np.histogram( df_tmp[key], bins = bin_count )
        hist1, bin_edges1 = np.histogram( df_tmp1[key], bins = bin_edges )
        hist2, bin_edges2 = np.histogram( df_tmp2[key], bins = bin_edges )
        #输出结果
        ret_df[key] = pd.Series(hist)
        ret_df1[key] = pd.Series(hist1)
        ret_df1[key] = ( ret_df1[key] - ret_df1[key].min() ) / ( ret_df1[key].max() - ret_df1[key].min() )
        ret_s1 = ret_s1.append(ret_df1[key], ignore_index=True)

        ret_df2[key] = pd.Series(hist2)
        ret_df2[key] = (ret_df2[key] - ret_df2[key].min()) / ( ret_df2[key].max() - ret_df2[key].min() )
        ret_s2 = ret_s2.append(ret_df2[key], ignore_index=True)

    return ret_df1.T,ret_s1,ret_df2.T,ret_s2

def CalcDistance(df1, df2, bin_count = 25 ):
    df_tmp1,s1,df_tmp2,s2 = GetHistogram(df1,df2,bin_count)
    ret = []
    for key in df_tmp1.T:
        ds1 = df_tmp1.T[key]
        ds2 = df_tmp2.T[key]
        s = CJDistance(ds1.tolist(),ds2.tolist())
        tmp = {}
        tmp["Feature"] = key
        tmp['Cosine'] = s.Cosine()
        tmp['Pearson'] = s.Pearson()
        tmp['Euclidean'] = s.Euclidean()
        tmp['EMD'] = s.EDM()
        tmp['KS'] = s.KSTest()[1]
        tmp['Manhattan'] = s.Manhattan()
        tmp['Minkowski'] = s.Minkowski()
        tmp['Jaccard'] = s.Jaccard()
        tmp['Entropy'] = s.Entropy()
        ret.append(tmp)
    return pd.DataFrame(ret)

In [85]:
df_tmp1 = pd.read_csv("%s%s"%(g_test_path,"0.csv"),index_col=0)
df_tmp2 = pd.read_csv("%s%s"%(g_sample_path,"1.csv"),index_col=0)
df_tmp3 = pd.read_csv("%s%s"%(g_sample_path,"6.csv"),index_col=0)
df01 = CalcDistance(df_tmp1,df_tmp2,25).sort_values(by='EMD').mean()
df06 = CalcDistance(df_tmp1,df_tmp3,25).sort_values(by='EMD').mean()

In [86]:
display(df01,df06)

Cosine      0.62699129
Pearson     0.55913474
Euclidean   0.94660990
EMD         0.02711956
KS          0.54327257
Manhattan   2.17846696
Minkowski   0.77550000
Jaccard     0.12032916
Entropy     0.41253746
dtype: float64

Cosine      0.60820644
Pearson     0.53593250
Euclidean   0.99303784
EMD         0.03907055
KS          0.53551681
Manhattan   2.31412248
Minkowski   0.83620000
Jaccard     0.11294121
Entropy     0.46523148
dtype: float64